# Beginning Python: Become a State Data Detective 🕵️

In this workshop, we will use a playful dataset about U.S. states to answer questions such as:

- Which states are best for a sunny brunch road trip?
- Which states are UFO hotspots?
- Does warmer weather seem connected to more sunny days?
- Which states are unusual compared with the rest?

We will build these answers with Python, pandas, NumPy, Matplotlib, and SciPy.

## Workshop roadmap

| Approx. time | Activity |
|---:|---|
| 0–8 min | Python warm-up and notebook setup |
| 8–20 min | Explore and filter the state data with pandas |
| 20–32 min | Use NumPy to create scores and categories |
| 32–44 min | Tell a story with Matplotlib |
| 44–52 min | Use SciPy to find unusual states and relationships |
| 52–60 min | Choose a winning road-trip destination |

## 1. Setup

Google Colab already includes the libraries we need. Run this cell once.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

plt.style.use("default")

DATA_URL = (
    "https://raw.githubusercontent.com/dparry13/hbll-data_training/refs/heads/main/"
    "Beginning%20Python%20Workshop/census_and_trends_data.csv"
)

census_df = pd.read_csv(DATA_URL)
print("The data loaded successfully!")

## 2. A quick Python warm-up

Python can act like a calculator, save information in variables, and call functions.

In [ ]:
favorite_number = 6
road_trip_snack = "waffles"

print(f"My favorite number is {favorite_number}.")
print(f"My road-trip snack is {road_trip_snack}.")

6 * 7
"waffles" == "tacos"

## 3. Meet the dataset

A pandas DataFrame is a table. Each row represents a state, and each column describes something about that state.

In [ ]:
print("Rows and columns:", census_df.shape)
print("First five states:")
display(census_df.head())

print("Available columns:")
print(census_df.columns.tolist())

### State passports

Select a few columns to create a compact state passport.

In [ ]:
census_df[[
    "state",
    "personality_profile",
    "avg_temp_f",
    "annual_sunny_days",
    "hiking",
    "capybara",
]]

### Challenge: find a sunny road-trip stop

Find states with at least 250 sunny days per year. Then select only the state name and sunshine columns.

In [ ]:
# Your code here

### One possible solution

In [ ]:
census_df.loc[
    census_df["annual_sunny_days"] >= 250,
    ["state", "annual_sunny_days"],
]

### UFO detectives 👽

Find states with at least 60 UFO sightings per 100,000 people. Is there a state you expected to see?

In [ ]:
ufo_hotspots = census_df[
    census_df["ufo_sightings_per_100k"] >= 60
][[
    "state",
    "ufo_sightings_per_100k",
    "personality_profile",
]]

ufo_hotspots

## 4. NumPy: create a brunch score 🧇

NumPy is especially useful when we want to perform the same numerical operation across many rows at once.

First, collect the three brunch columns as a NumPy array.

In [ ]:
trend_columns = ["waffles", "pancakes", "donuts"]
trend_array = census_df[trend_columns].to_numpy()

print("Array shape:", trend_array.shape)
print("First state's brunch scores:", trend_array[0])

Use NumPy's sum and mean to calculate a total brunch score and an average brunch score for every state.

In [ ]:
census_df = census_df.assign(
    brunch_score=trend_array.sum(axis=1),
    brunch_average=trend_array.mean(axis=1),
)

census_df[[
    "state", "waffles", "pancakes", "donuts",
    "brunch_score", "brunch_average",
]].sort_values("brunch_score", ascending=False).head(10)

### NumPy challenge: define “sunny side”

Use the median number of sunny days as the cutoff. Create a new column called sunshine_label with np.where.

In [ ]:
sunny_median = np.median(census_df["annual_sunny_days"])

census_df = census_df.assign(
    sunshine_label=np.where(
        census_df["annual_sunny_days"] >= sunny_median,
        "sunny side",
        "cloudier side",
    )
)

census_df[["state", "annual_sunny_days", "sunshine_label"]].head()

## 5. Matplotlib: let the data tell a story 📊

A visualization is useful when it helps us notice a pattern. Start with the distribution of average temperatures.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(census_df["avg_temp_f"], bins=10, edgecolor="white")
ax.set_title("How warm are the states?")
ax.set_xlabel("Average temperature (°F)")
ax.set_ylabel("Number of states")

plt.show()

### Temperature versus sunshine

Each dot is a state. Look for clusters, unusual points, and whether the relationship looks strong or weak.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    census_df["avg_temp_f"],
    census_df["annual_sunny_days"],
    alpha=0.75,
)

ax.set_title("Temperature and annual sunshine")
ax.set_xlabel("Average temperature (°F)")
ax.set_ylabel("Annual sunny days")

plt.show()

### Personality profiles and sunshine

Group the states by personality profile and make a bar chart of average sunny days.

In [ ]:
profile_summary = (
    census_df
    .groupby("personality_profile", as_index=False)
    .agg(average_sunny_days=("annual_sunny_days", "mean"))
    .sort_values("average_sunny_days")
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    profile_summary["personality_profile"],
    profile_summary["average_sunny_days"],
)
ax.set_title("Which personality profile gets the most sunshine?")
ax.set_ylabel("Average sunny days")
ax.tick_params(axis="x", rotation=45)
plt.show()

## 6. SciPy: find unusual states 🔎

A z-score tells us how many standard deviations a value is above or below the average. Large positive values are unusually high; large negative values are unusually low.

In [ ]:
ufo_z_scores = stats.zscore(census_df["ufo_sightings_per_100k"])

ufo_outliers = census_df.assign(ufo_z_score=ufo_z_scores)

ufo_outliers[[
    "state", "ufo_sightings_per_100k", "ufo_z_score",
]].sort_values("ufo_z_score", ascending=False).head(5)

### SciPy: quantify the scatterplot

The Pearson correlation measures the direction and strength of a linear relationship. It does not prove that one variable causes the other.

In [ ]:
correlation, p_value = stats.pearsonr(
    census_df["avg_temp_f"],
    census_df["annual_sunny_days"],
)

print(f"Correlation: {correlation:.2f}")
print(f"P-value: {p_value:.4f}")

## 7. Final challenge: choose a road-trip champion 🚗

You are planning a sunny outdoor brunch road trip. A qualifying state must:

- have at least 225 sunny days;
- have at least one national park; and
- have a hiking trend score of at least 20.

Create a road_trip_score that combines brunch, hiking, sunshine, parks, and capybara friendliness. Because you have a pet capybara, give states with higher capybara interest a bonus. Keep major zoos as a separate travel feature. This score is for practice—not a scientific measure!

In [ ]:
road_trip_states = census_df[
    (census_df["annual_sunny_days"] >= 225)
    & (census_df["national_parks_count"] >= 1)
    & (census_df["hiking"] >= 20)
].copy()

road_trip_states["zoo_bonus"] = np.where(
    road_trip_states["has_major_zoo"],
    15,
    0,
)

road_trip_states["capybara_friendly_rating"] = road_trip_states["capybara"]

road_trip_states["road_trip_score"] = (
    road_trip_states["brunch_score"]
    + road_trip_states["hiking"]
    + road_trip_states["annual_sunny_days"] / 10
    + road_trip_states["national_parks_count"] * 5
    + road_trip_states["capybara_friendly_rating"] / 2
    + road_trip_states["zoo_bonus"]
)

road_trip_states[[
    "state", "annual_sunny_days", "hiking",
    "national_parks_count", "has_major_zoo", "capybara",
    "zoo_bonus", "capybara_friendly_rating", "road_trip_score",
]].sort_values("road_trip_score", ascending=False)

In [ ]:
top_states = road_trip_states.sort_values("road_trip_score").tail(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_states["state"], top_states["road_trip_score"])
ax.set_title("Road-trip champions")
ax.set_xlabel("Practice road-trip score")

plt.show()

### Share your conclusion

Answer in a sentence or two:

1. Which state would you choose?
2. Which column most influenced your choice?
3. What would you add to the dataset to make the recommendation better?

## What you practiced

- Python variables, expressions, and Boolean comparisons
- pandas selection, filtering, assignment, grouping, and sorting
- NumPy arrays, vectorized calculations, percentiles, and where
- Matplotlib histograms, scatterplots, and bar charts
- SciPy z-scores and Pearson correlation
- Turning a question into a reproducible analysis